<a href="https://colab.research.google.com/github/jc020230/practiceai/blob/main/1_pandas_data_structures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pandas Data Structures Overview
In this section, we will discuss the `Series`, `Index`, and `DataFrame` classes. To do so, we will read in a snippet of the CSV file we will work with later. Don't worry about that part yet, though.

## About the Data
In this notebook, we will be working with 5 rows from the earthquake data collected over September 18, 2018 - October 13, 2018 (obtained from the US Geological Survey (USGS) using the [USGS API](https://earthquake.usgs.gov/fdsnws/event/1/))

## Working with NumPy Arrays
Let's read in a short CSV file (using `numpy`) for some sample data.

요약
- numpy(넘파이): 숫자 계산용 도구. 배열(array)이라는 걸 사용
- pandas(판다스): 표의 데이터용 도구.

- pandas의 3가지 핵심 구조:
  - Series = 표에서 열(column) 하나
  - Index = 각 행(또는 열)에 붙은 이름표 (기본은 0, 1, 2, ... 번호)
  - DataFrame = 표 전체 (엑셀 시트 하나라고 생각하면 됨)

- 앞부분(numpy)은 "numpy만 쓰면 왜 불편한가"를 보여주는 거라서 코드가 어려우면 넘어가도 됨. Series가 핵심임

In [ ]:
# numpy(넘파이): 숫자 계산을 빠르게 해주는 파이썬에 내장되어 있는 도구
# import numpy as np → numpy를 꺼내서 앞으로 np라는 짧은 이름으로 부를게! 라는 뜻
# (pandas도 pd로 줄여씀. import pandas as pd
import numpy as np

# genfromtxt: 텍스트 파일(csv)을 읽어서 numpy 배열(array)로 만들어주는 명령어
# 'data/example_data.csv' → 읽을 파일 위치
# delimiter=';' → 이 파일은 값들이 ;(세미콜론)으로 구분되어 있음
# names=True → 첫 줄을 열(column) 이름으로 써줘
# dtype=None → data type 줄여서 dtype, 데이터 타입은 알아서 판단해줘
# encoding='UTF' → 글자 읽는 방식 (특수문자 안 깨지게)
data = np.genfromtxt(
    'data/example_data.csv', delimiter=';',
    names=True, dtype=None, encoding='UTF'
)
# 변수 이름만 치면 내용이 출력됨 (셀의 마지막 줄은 print 없이도 자동 출력)
# 결과: 지진 5개의 (시간, 장소, 규모종류, 규모, 경보색, 쓰나미여부) 정보
data


array([('2018-10-13 11:10:23.560', '262km NW of Ozernovskiy, Russia', 'mww', 6.7, 'green', 1),
       ('2018-10-13 04:34:15.580', '25km E of Bitung, Indonesia', 'mww', 5.2, 'green', 0),
       ('2018-10-13 00:13:46.220', '42km WNW of Sola, Vanuatu', 'mww', 5.7, 'green', 0),
       ('2018-10-12 21:09:49.240', '13km E of Nueva Concepcion, Guatemala', 'mww', 5.7, 'green', 0),
       ('2018-10-12 02:52:03.620', '128km SE of Kimbe, Papua New Guinea', 'mww', 5.6, 'green', 1)],
      dtype=[('time', '<U23'), ('place', '<U37'), ('magType', '<U3'), ('mag', '<f8'), ('alert', '<U5'), ('tsunami', '<i8')])

쓰나미 데이터니까
- time은 발생 시간
- place는 발생 장소
- magtype은 얼마나 심한지
- mag도 얼마나 심한지 알려주는 지표

We can find the dimensions with the `shape` attribute:

In [ ]:
# shape: 모양(크기). 몇 행 몇 열이냐?
# (5,) → 5개짜리라는 뜻. (5,)로 뒤가 비어있는 건 행렬중 행의 개념만 있고 열의 개념이 없어서
data.shape


(5,)

We can find the data types with the `dtype` attribute:

In [ ]:
# dtype: data type(데이터 타입). 각 열이 어떤 종류의 값인지
# <U23 → 최대 23글자 문자열(U = Unicode 문자), <f8 → 소수(float), <i8 → 정수(integer)
# time, place, magType, alert는 문자열 / mag는 소수 / tsunami는 정수
data.dtype


dtype([('time', '<U23'), ('place', '<U37'), ('magType', '<U3'), ('mag', '<f8'), ('alert', '<U5'), ('tsunami', '<i8')])

Each of the entries in the array is a row from the CSV file. NumPy arrays contain a single data type (unlike lists, which allow mixed types); this allows for fast, vectorized operations. When we read in the data, we got an array of `numpy.void` objects, which are created to store flexible types. This is because NumPy has to store several different data types per row: four strings, a float, and an integer. This means we can't take advantage of the performance improvements NumPy provides for single data type objects.

Say we want to find the maximum magnitude&mdash;we can use a **[list comprehension](https://www.python.org/dev/peps/pep-0202/)** to select the third index of each row, which is represented as a `numpy.void` object. This makes a list, meaning that we can take the maximum using the `max()` function:

In [ ]:
%%timeit
# %%timeit은 신경 안 써도 됨

# [row[3] for row in data] → data의 각 행(row)에서 3번째 칸(0부터 세니까 4번째) = mag(규모)만 뽑아 리스트로 만듦
# 이런 문법을 list comprehension이라고 함. for문을 한 줄로 줄여 쓴 거임
# max(...) → 그 리스트에서 최댓값 = 가장 큰 규모(6.7)
max([row[3] for row in data])


9.74 µs ± 177 ns per loop (mean ± std. dev. of 7 runs, 10000 loops each)


If we, instead, create a NumPy array for each column, this operation is much easier (and more efficient) to perform. We can use a **[dictionary comprehension](https://www.python.org/dev/peps/pep-0274/)** to make a dictionary where the keys are the column names and the values are NumPy arrays of the data:

In [ ]:
# 위 방식은 '행' 단위로 묶여 있어서 불편함. 이번엔 '열(column)' 단위로 다시 정리해보는 거임
# 딕셔너리(dictionary) = {키: 값} 형태. 여기서 키는 열 이름, 값은 그 열의 값들이 든 numpy 배열
# enumerate(data.dtype.names) → 열 이름들(time, place, ...)을 번호(i)와 함께 하나씩 꺼내줌
#   i=0, col='time' / i=1, col='place' / i=2, col='magType' ... 이런 식
# np.array([row[i] for row in data]) → 모든 행에서 i번째 값만 뽑아서 배열로 만듦
# {키: 값 for ...} 형태로 쓴 걸 dictionary comprehension이라고 함 (list comprehension의 딕셔너리 버전)
array_dict = {
    col: np.array([row[i] for row in data])
    for i, col in enumerate(data.dtype.names)
}
# 결과: {'time': [시간들...], 'place': [장소들...], 'mag': [6.7, 5.2, ...], ...} 이렇게 나옴
array_dict


{'time': array(['2018-10-13 11:10:23.560', '2018-10-13 04:34:15.580',
        '2018-10-13 00:13:46.220', '2018-10-12 21:09:49.240',
        '2018-10-12 02:52:03.620'], dtype='<U23'),
 'place': array(['262km NW of Ozernovskiy, Russia', '25km E of Bitung, Indonesia',
        '42km WNW of Sola, Vanuatu',
        '13km E of Nueva Concepcion, Guatemala',
        '128km SE of Kimbe, Papua New Guinea'], dtype='<U37'),
 'magType': array(['mww', 'mww', 'mww', 'mww', 'mww'], dtype='<U3'),
 'mag': array([6.7, 5.2, 5.7, 5.7, 5.6]),
 'alert': array(['green', 'green', 'green', 'green', 'green'], dtype='<U5'),
 'tsunami': array([1, 0, 0, 0, 1])}

Grabbing the maximum magnitude is now simply a matter of selecting the `mag` key and calling the `max()` method. This is nearly twice as fast as the list comprehension implementation when dealing with just 5 entries, imagine how much worse the first attempt will perform on large data sets:

In [ ]:
%%timeit
# array_dict['mag'] → 딕셔너리에서 'mag' 키의 값 = 규모들만 들어있는 배열
# .max() → 그 배열의 최댓값
# 위에서 for문 돌린 것보다 훨씬 빠름. numpy 배열은 한 종류 값만 있으면 계산이 빨라지기 때문
# 이래서 numpy가 좋다~~ 서술형 나오면 이거 쓰면 됨
array_dict['mag'].max()


5.22 µs ± 100 ns per loop (mean ± std. dev. of 7 runs, 10000 loops each)


However, this representation has other issues. Say we wanted to grab all the information for the earthquake with the maximum magnitude, how would we go about that? We would need to find the index of the maximum and then for each of the keys in the dictionary grab that index:

In [ ]:
# "가장 규모가 큰 지진의 정보를 전부 가져오고 싶다" → 이 방식으로는 좀 번거로움
# array_dict['mag'].argmax() → arg(위치) + max(최대) = 최댓값이 있는 '위치(번호)'. 여기서는 0 (첫 번째 지진이 6.7로 최대)
# array_dict.items() → 딕셔너리의 (키, 값) 쌍을 하나씩 꺼냄
# value[...] → 각 열(value)에서 그 위치의 값을 하나씩 뽑음
# 결과: 첫 번째 지진의 정보가 다 나오긴 하는데, 숫자였던 6.7과 1이 문자열('6.7', '1')로 바뀌어버림
# → numpy 배열은 한 종류만 담을 수 있어서 전부 문자열로 통일된 거임. 이래서 pandas가 필요함!
np.array([
    value[array_dict['mag'].argmax()]
    for key, value in array_dict.items()
])


array(['2018-10-13 11:10:23.560', '262km NW of Ozernovskiy, Russia',
       'mww', '6.7', 'green', '1'], dtype='<U31')

The result is now a NumPy array of strings (our numeric values were converted), and we are now in the format from earlier. Also, consider trying to sort the data by magnitude from smallest to largest. In the first representation, we would have to sort the rows by examining the 3rd index. With the second representation, we would have to determine the order for the indices from the `mag` column, and then sort all the other arrays with those same indices. Clearly, working with several NumPy arrays of different data types at once is a bit cumbersome. However, `pandas` builds on top of NumPy arrays to make this easier. Let's start our exploration of `pandas` with an overview of the data structures.

## `Series`
The `Series` class provides a data structure for arrays of a single type with some additional functionality.

In [ ]:
# pandas(판다스): 표(엑셀 같은) 데이터를 다루는 파이썬 도구. pd라는 짧은 이름으로 부름
import pandas as pd

# Series: pandas의 한 줄짜리 열(column). 엑셀에서 열 하나만 뚝 떼어온 거라고 생각하면 됨
# pd.Series(데이터, name='이름') → array_dict의 place 배열로 시리즈를 만들고 이름을 'place'로 붙임
place = pd.Series(array_dict['place'], name='place')
# 왼쪽에 0~4 번호가 붙어서 나옴 → 이게 index(인덱스). 각 행의 이름표 같은 거임
# 맨 아래 Name: place, dtype: object → 이름은 place, 타입은 object (문자열은 pandas에서 object로 표시됨)
place


0          262km NW of Ozernovskiy, Russia
1              25km E of Bitung, Indonesia
2                42km WNW of Sola, Vanuatu
3    13km E of Nueva Concepcion, Guatemala
4      128km SE of Kimbe, Papua New Guinea
Name: place, dtype: object

Here are some commonly used attributes with `Series` objects:

|Attribute | Returns |
| --- | --- |
| `name` | The name of the `Series` object |
| `dtype` | The data type of the `Series` object |
| `shape` | Dimensions of the `Series` object in a tuple of the form `(number of rows,)` |
| `index` | The `Index` object that is part of the `Series` object |
| `values` | The data in the `Series` object |

For the most part, `pandas` objects use NumPy arrays for their internal data representations. However, for some data types, `pandas` builds upon NumPy to create its own [arrays](https://pandas.pydata.org/pandas-docs/stable/reference/arrays.html). For this reason, depending on the data type, `values` can either be a `pandas.array` or `numpy.array` object. Therefore, if we need to ensure we get a specific type back, then it is recommended to use the `array` attribute or `to_numpy()` method, respectively, instead of `values`.

Now let's see some examples using these attributes.

### Getting the name of the series
The NumPy array held the name of the data in the `dtype` attribute; here, we can access it directly:

In [ ]:
# .name → 시리즈에 붙인 이름. 위에서 name='place'라고 했으니 'place'가 나옴
place.name


'place'

### Getting the data type
A `Series` object holds a single data type. Here it is `'O'` for object.

In [ ]:
# .dtype → 데이터 타입. 'O'는 Object(오브젝트)의 약자로, 문자열이 들어있으면 이렇게 나옴
place.dtype


dtype('O')

### Getting the dimensions of the series
Just as with NumPy, we can use `shape` to get the dimensions as `(rows, columns)`. `Series` objects are a single column, so they only have values for the rows dimension.

In [ ]:
# .shape → 크기. (5,) = 5행. 시리즈는 열이 하나라서 뒤가 비어있음
place.shape


(5,)

### Isolating the values from the series
This `Series` object is storing its values as a NumPy array:

In [ ]:
# .values → 인덱스(번호표) 떼고 알맹이 값만 numpy 배열로 꺼냄
# pandas는 numpy가 내장되어 있어 numpy를 쓰고 있어서 이렇게 numpy 배열이 나오는 거임
place.values


array(['262km NW of Ozernovskiy, Russia', '25km E of Bitung, Indonesia',
       '42km WNW of Sola, Vanuatu',
       '13km E of Nueva Concepcion, Guatemala',
       '128km SE of Kimbe, Papua New Guinea'], dtype=object)

## `Index`
The addition of the `Index` class makes the `Series` class more powerful than a NumPy array. We can get the index from the `index` attribute of a `Series` object:

In [ ]:
# .index → 시리즈의 인덱스(행 이름표)만 따로 꺼냄
place_index = place.index
# RangeIndex(start=0, stop=5, step=1) → 0부터 5 직전(4)까지 1씩 증가하는 번호. 즉 0,1,2,3,4
# 우리가 따로 이름을 안 정해줬으니 자동으로 번호가 붙은 거임 (python_101의 range(5)와 같은 개념)
place_index


RangeIndex(start=0, stop=5, step=1)

As with `Series` objects, we can access the underlying data via the `values` attribute. Note that this `Index` object is also built on top of a NumPy array:

In [ ]:
# 인덱스도 .values로 알맹이만 꺼낼 수 있음 → [0, 1, 2, 3, 4]
place_index.values


array([0, 1, 2, 3, 4])

Here are some commonly used attributes with `Index` objects:

|Attribute | Returns |
| --- | --- |
| `name` | The name of the `Index` object |
| `dtype` | The data type of the `Index` object |
| `shape` | Dimensions of the `Index` object |
| `values` | The data in the `Index` object |
| `is_unique` | Check if the `Index` object has all unique values |

We can check the type of the underlying data, just like with a `Series` object:

In [ ]:
# 인덱스의 데이터 타입. 0~4 정수니까 int64 (int = integer 정수, 64 = 저장 크기. 숫자는 신경 안 써도 됨)
place_index.dtype


dtype('int64')

Same for the dimensions:

In [ ]:
# 인덱스의 크기. 5개 (0, 1, 2, 3, 4) 니까
place_index.shape


(5,)

We can check if the values are unique:

In [ ]:
# is_unique → 값들이 전부 다른가(중복 없나)? 0,1,2,3,4 다 다르니까 True
# 인덱스는 이름표니까 중복이 없는지 확인하는 게 중요함
place_index.is_unique


True

With NumPy we can perform arithmetic operations element-wise between arrays:

In [ ]:
# numpy 배열끼리 더하면 같은 위치끼리 더해짐 (element-wise = 요소별 연산)
# [1, 1, 1] + [-1, 0, 1] → [1+(-1), 1+0, 1+1] = [0, 1, 2]
np.array([1, 1, 1]) + np.array([-1, 0, 1])


array([0, 1, 2])

Pandas supports this as well, and the index determines how element-wise operations are performed. With addition, only the matching indices are summed:

In [ ]:
# np.linspace(0, 10, num=5) → 0부터 10까지 똑같은 간격으로 5개 숫자 만듦: 0, 2.5, 5, 7.5, 10
numbers = np.linspace(0, 10, num=5) # makes numpy array([0, 2.5, 5, 7.5, 10])
# x: 인덱스를 안 정했으니 자동으로 0,1,2,3,4
x = pd.Series(numbers) # index is [0, 1, 2, 3, 4]
# y: 같은 숫자인데 인덱스를 1,2,3,4,5로 직접 정해줌
y = pd.Series(numbers, index=pd.Index([1, 2, 3, 4, 5]))
# pandas는 '위치'가 아니라 '인덱스 이름표'가 같은 것끼리 더함! (numpy와 다른 점)
# 인덱스 1: x의 2.5 + y의 0 = 2.5 / 인덱스 2: 5 + 2.5 = 7.5 / 인덱스 3: 7.5 + 5 = 12.5 ...
# 인덱스 0은 x에만 있고 y에는 없음 → 짝이 없어서 NaN
# 인덱스 5는 y에만 있음 → 역시 NaN
# NaN = Not a Number, 값 없음/빈칸이라는 뜻.
x + y


0     NaN
1     2.5
2     7.5
3    12.5
4    17.5
5     NaN
dtype: float64

We aren't limited to the integer indices of list-like structures, and we can label our rows. The labels can be altered at any time and be things like dates or even another column. In chapter 3, we will discuss how to perform some operations on the index in order to change it. Then, in chapter 4, we will use the index for operations merging data and aggregating it.

## `DataFrame`
Having a `Series` object for each column is an improvement over the NumPy representation; however, we still have the same problem when wanting to sort based on a value or grab an entire row out. The `DataFrame` gives us a representation of a table formed from many `Series` objects that form the columns and a shared `Index` object that labels the rows. We can create a `DataFrame` object from either of the NumPy representations we were working with earlier (we could also make a `Series` object for each column, but there is no need to do so):

In [ ]:
# DataFrame(데이터프레임): pandas의 '표'. 엑셀 시트 하나라고 생각하면 됨
# 시리즈(열)가 여러 개 모여서 하나의 표가 된 것
# pd.DataFrame(딕셔너리) → {열이름: 값들} 딕셔너리를 넣으면 각 키가 열 이름이 됨
df = pd.DataFrame(array_dict)

# this will also work with the first representation
# (맨 처음 numpy로 읽은 data를 바로 넣어도 됨)
# df = pd.DataFrame(data)

# df는 DataFrame의 줄임말로 보통 가장 많이 쓰는 변수 이름
# 결과: 왼쪽에 인덱스 0~4, 위에 열 이름 6개(time, place, magType, mag, alert, tsunami)가 있는 표
df


,time,place,magType,mag,alert,tsunami
0,2018-10-13 11:10:23.560,"262km NW of Ozernovskiy, Russia",mww,6.7,green,1
1,2018-10-13 04:34:15.580,"25km E of Bitung, Indonesia",mww,5.2,green,0
2,2018-10-13 00:13:46.220,"42km WNW of Sola, Vanuatu",mww,5.7,green,0
3,2018-10-12 21:09:49.240,"13km E of Nueva Concepcion, Guatemala",mww,5.7,green,0
4,2018-10-12 02:52:03.620,"128km SE of Kimbe, Papua New Guinea",mww,5.6,green,1


We can check the type of the underlying data with `dtypes` (note that it is not `dtype` as with `Series` and `Index` objects since each column will have its own data type):

In [ ]:
# dtypes → 각 열의 데이터 타입 (시리즈는 dtype, 데이터프레임은 열이 여러 개라 s 붙여서 dtypes)
# mag는 float64(소수), tsunami는 int64(정수), 나머지는 object(문자열)
# 앞에서 numpy는 전부 문자열로 바꿔버렸는데, pandas는 열마다 타입을 따로 유지함! 이게 장점
# df.dtypes니까 df(데이터프레임)의 데이터 타입을 물어보는 코드
df.dtypes


time        object
place       object
magType     object
mag        float64
alert       object
tsunami      int64
dtype: object

We can get the underlying data with the `values` attribute. Note that this looks very similar to our initial NumPy representation:

In [ ]:
# .values → 인덱스, 열 이름 떼고 알맹이만 numpy 배열로. 맨 처음 봤던 numpy 모양과 비슷함
# df.values 니까 df의 값들을 numpy 배열로 바꾼다는 뜻.
df.values


array([['2018-10-13 11:10:23.560', '262km NW of Ozernovskiy, Russia',
        'mww', 6.7, 'green', 1],
       ['2018-10-13 04:34:15.580', '25km E of Bitung, Indonesia', 'mww',
        5.2, 'green', 0],
       ['2018-10-13 00:13:46.220', '42km WNW of Sola, Vanuatu', 'mww',
        5.7, 'green', 0],
       ['2018-10-12 21:09:49.240',
        '13km E of Nueva Concepcion, Guatemala', 'mww', 5.7, 'green', 0],
       ['2018-10-12 02:52:03.620', '128km SE of Kimbe, Papua New Guinea',
        'mww', 5.6, 'green', 1]], dtype=object)

We can isolate the columns with the `columns` attribute. Notice that the columns are actually an `Index` object just on a different axis (columns are the horizontal index while rows are the vertical index).

In [ ]:
# .columns → 열 이름들. 얘도 Index 타입임
# 행 방향 이름표 = index, 열 방향 이름표 = columns.
df.columns


Index(['time', 'place', 'magType', 'mag', 'alert', 'tsunami'], dtype='object')

Here are some commonly used attributes:

|Attribute | Returns |
| --- | --- |
| `dtypes` | The data types of each column |
| `shape` | Dimensions of the `DataFrame` object in a tuple of the form `(number of rows, number of columns)` |
| `index` | The `Index` object along the rows of the `DataFrame` object |
| `columns` | The name of the columns (as an `Index` object) |
| `values` | The data in the `DataFrame` object |
| `empty` | Check if the `DataFrame` object is empty |

The `Index` object along the rows of the dataframe can be accessed via the `index` attribute (just as with `Series` objects):

In [ ]:
# .index → 행 이름표. 0~4 자동 번호
df.index


RangeIndex(start=0, stop=5, step=1)

As with both `Series` and `Index` objects, we can get the dimensions of the dataframe with the `shape` attribute. The result is of the form `(nrows, ncols)`. Our dataframe has 5 rows and 6 columns:

In [ ]:
# .shape → (행 개수, 열 개수) = (5, 6). 지진 5개, 정보 6가지
# 데이터 받으면 제일 먼저 확인하는 것 중 하나
df.shape


(5, 6)

Note that we can also perform arithmetic on dataframes. Pandas will only perform the operation when both the index and column match. Here, we demonstrate addition. Since addition with strings means concatenation, `pandas` concatenated the string columns (`time`, `place`, `magType`, and `alert`) across dataframes. The numeric columns (`mag` and `tsunami`) were summed:

In [ ]:
# 데이터프레임끼리 더하면 인덱스와 열 이름이 같은 칸끼리 더해짐
# 숫자 열(mag, tsunami)은 진짜 더해짐: 6.7 + 6.7 = 13.4
# 문자열 열(time, place 등)은 + 하면 이어붙이기(concat)가 됨: 'mww' + 'mww' = 'mwwmww'
df + df


,time,place,magType,mag,alert,tsunami
0,2018-10-13 11:10:23.5602018-10-13 11:10:23.560,"262km NW of Ozernovskiy, Russia262km NW of Oze...",mwwmww,13.4,greengreen,2
1,2018-10-13 04:34:15.5802018-10-13 04:34:15.580,"25km E of Bitung, Indonesia25km E of Bitung, I...",mwwmww,10.4,greengreen,0
2,2018-10-13 00:13:46.2202018-10-13 00:13:46.220,"42km WNW of Sola, Vanuatu42km WNW of Sola, Van...",mwwmww,11.4,greengreen,0
3,2018-10-12 21:09:49.2402018-10-12 21:09:49.240,"13km E of Nueva Concepcion, Guatemala13km E of...",mwwmww,11.4,greengreen,0
4,2018-10-12 02:52:03.6202018-10-12 02:52:03.620,"128km SE of Kimbe, Papua New Guinea128km SE of...",mwwmww,11.2,greengreen,2


<hr>
<div>
    <a href="../ch_01/introduction_to_data_analysis.ipynb">
        <button style="float: left;">&#8592; Chapter 1</button>
    </a>
    <a href="./2-creating_dataframes.ipynb">
        <button style="float: right;">Next Notebook &#8594;</button>
    </a>
</div>
<br>
<hr>